# common (library notebook)

Shared code, loaded by every other notebook with `%run ./common.ipynb`; it only defines things.

* paths (`ROOT`, `DATA`, `RUNS`) and the **writer-independent split**
* dataset + augmentation, model, prediction, metrics (AUC, EER, FAR, FRR)
* writer-dependent SVM helpers (used by `wd_svm.ipynb` and `gradcam_verifier.ipynb`)
* the Grad-CAM overlay

Open the notebooks from the `src/` folder: paths are resolved relative to it.

In [1]:
%run ./preprocess.ipynb

In [2]:
import os

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.svm import SVC
from concurrent.futures import ThreadPoolExecutor
from torch.utils.data import Dataset, default_collate
from torchvision.models import EfficientNet_B2_Weights, efficientnet_b2


def _find_root():
    for p in (os.getcwd(), os.path.dirname(os.getcwd())):
        if os.path.isdir(os.path.join(p, "src")):
            return p
    raise FileNotFoundError("Open the notebooks from the project's src/ folder.")


ROOT = _find_root()
DATA = os.path.join(ROOT, "data")
RUNS = os.path.join(ROOT, "runs")
MEAN = np.array([0.485, 0.456, 0.406], np.float32).reshape(3, 1, 1)
STD = np.array([0.229, 0.224, 0.225], np.float32).reshape(3, 1, 1)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Writer-independent split

In [3]:
def writer_split(fold=0, n_folds=5, n_val=8, seed=42):
    """Writer-independent k-fold split: no writer is in more than one of train/val/test.

    Writers are shuffled once (seed), cut into n_folds chunks; chunk `fold` is
    the test set and n_val writers of the rest are held out for validation
    (model selection + decision threshold).
    """
    meta = pd.read_csv(os.path.join(DATA, "meta.csv"))
    writers = np.array(sorted(meta.writer.unique()))
    np.random.default_rng(seed).shuffle(writers)
    chunks = np.array_split(writers, n_folds)
    test_w = chunks[fold]
    rest = np.concatenate([c for i, c in enumerate(chunks) if i != fold])
    val_w, train_w = rest[:n_val], rest[n_val:]
    idx = lambda ws: np.where(meta.writer.isin(ws))[0]
    return meta, idx(train_w), idx(val_w), idx(test_w)


def load_gray(mode):
    return np.load(os.path.join(DATA, f"cache_{mode}_gray.npy"), mmap_mode="r")

## Dataset and augmentation

In [4]:
def augment(canon, rng):
    """Random geometry + stroke-width jitter on the canonical (white-bg) signature."""
    h, w = canon.shape
    ang, sc = rng.uniform(-6, 6), rng.uniform(0.82, 1.0)       # shrink only: wide signatures fill the canvas
    M = cv2.getRotationMatrix2D((w / 2, h / 2), ang, sc)
    M[0, 1] += np.tan(np.radians(rng.uniform(-8, 8)))        # shear
    M[:, 2] += (rng.uniform(-10, 10), rng.uniform(-8, 8))     # translate
    img = cv2.warpAffine(canon, M, (w, h), flags=cv2.INTER_LINEAR, borderValue=255)
    r = rng.random()
    if r < 0.25:                                              # thicker strokes
        img = cv2.erode(img, cv2.getStructuringElement(cv2.MORPH_CROSS, (3, 3)))
    elif r < 0.5:                                             # thinner strokes
        img = cv2.dilate(img, cv2.getStructuringElement(cv2.MORPH_CROSS, (3, 3)))
    return np.where(img < 128, 0, 255).astype(np.uint8)       # back to binary


def to_input(canon, rng=None):
    """Canonical image -> (optional augmentation) -> 4-stage pipeline -> normalised 3-ch array."""
    if rng is not None:
        canon = augment(canon, rng)
    return (stages_to_channels(canon).astype(np.float32) / 255.0 - MEAN) / STD


class SignatureSet(Dataset):
    """(image, label) pairs. Channels: 0 median-filtered, 1 high-boost, 2 Sobel."""

    def __init__(self, G, labels, train):
        self.G, self.y, self.train = G, labels.astype(np.float32), train

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        rng = np.random.default_rng() if self.train else None
        return torch.from_numpy(to_input(np.asarray(self.G[i]), rng)), torch.tensor(self.y[i])


class SingleSet(Dataset):
    """Images only (no labels, no augmentation), for embedding."""

    def __init__(self, G, idx):
        self.G, self.idx = G, idx

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        return torch.from_numpy(to_input(np.asarray(self.G[self.idx[i]])))


class ThreadLoader:
    """Batches a Dataset using a thread pool, and prepares the next batch while the GPU works.

    Replaces torch's DataLoader here: in a notebook on Windows, DataLoader worker *processes*
    can't use classes defined in the notebook. OpenCV releases the GIL, so threads run the
    preprocessing in parallel just as well.
    """

    def __init__(self, dataset, batch_size, shuffle=False, drop_last=False, workers=8):
        self.dataset, self.bs, self.shuffle, self.drop_last = dataset, batch_size, shuffle, drop_last
        self.workers = min(workers, os.cpu_count() or 1)

    def __len__(self):
        n = len(self.dataset)
        return n // self.bs if self.drop_last else -(-n // self.bs)

    def __iter__(self):
        n = len(self.dataset)
        order = np.random.permutation(n) if self.shuffle else np.arange(n)
        batches = [order[i:i + self.bs] for i in range(0, n, self.bs)][:len(self)]
        with ThreadPoolExecutor(self.workers) as ex:
            submit = lambda b: [ex.submit(self.dataset.__getitem__, int(i)) for i in b]
            pending = submit(batches[0]) if batches else []
            for k in range(len(batches)):
                items = [f.result() for f in pending]
                if k + 1 < len(batches):
                    pending = submit(batches[k + 1])        # prefetch the next batch
                yield default_collate(items)

## Model, prediction, learning-rate schedule

In [5]:
def build_model(pretrained=True, dropout=0.5):
    m = efficientnet_b2(weights=EfficientNet_B2_Weights.IMAGENET1K_V1 if pretrained else None)
    m.classifier[0].p = dropout
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, 1)   # logit: forged
    return m


@torch.no_grad()
def predict(model, loader, device):
    """Returns raw logits (not probabilities: sigmoid saturates to exactly 1.0
    for confident predictions, which breaks threshold selection)."""
    model.eval()
    zs, ys = [], []
    for x, y in loader:
        with torch.autocast(device.type, enabled=device.type == "cuda"):
            zs.append(model(x.to(device)).float().squeeze(1).cpu())
        ys.append(y)
    return torch.cat(zs).numpy(), torch.cat(ys).numpy()


def warmup_cosine(opt, warm, steps):
    """Linear warm-up for `warm` steps, then cosine decay to 0 at `steps`."""
    import math
    return torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else
        0.5 * (1 + math.cos(math.pi * min(s - warm, steps - warm) / max(steps - warm, 1))))

## Metrics

In [6]:
def eer_threshold(y, s):
    """Score threshold where FAR (forgeries accepted) == FRR (genuine rejected)."""
    fpr, tpr, thr = roc_curve(y, s)                  # positive class = forged
    fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr))
    return float(thr[i]), float((fpr[i] + fnr[i]) / 2)


def metrics(y, s, thr):
    pred = s >= thr
    forged, genuine = y == 1, y == 0
    far = float((~pred[forged]).mean())              # forged called genuine  -> fraud gets through
    frr = float(pred[genuine].mean())                # genuine called forged  -> real customer blocked
    _, eer = eer_threshold(y, s)
    return {"acc": float((pred == y).mean()), "auc": float(roc_auc_score(y, s)),
            "far": far, "frr": frr, "eer": eer, "thr": thr}

## Writer-dependent SVM verification

For a writer `w`: an RBF SVM on CNN embeddings, with `w`'s first `n_ref` genuine signatures as positives and
genuine signatures of *other* (training) writers as negatives. No skilled forgery is used to build it.
The forgery score of a questioned signature is `-decision_function`.

In [7]:
@torch.no_grad()
def embed_all(model, G, device, bs=32):
    model.eval()
    feats = []
    for x in ThreadLoader(SingleSet(G, np.arange(len(G))), batch_size=bs):
        f = model.avgpool(model.features(x.to(device)))
        feats.append(torch.flatten(f, 1).float().cpu())
    return torch.cat(feats).numpy()


def cnn_features(tag, mode, fold, device=DEVICE):
    """L2-normalised embeddings of all images. tag = imagenet | binary | writerid | writeridF;
    fine-tuned checkpoints are per fold (runs/<tag>_fold<k>/best.pt). Cached in runs/features/."""
    path = os.path.join(RUNS, "features", f"{tag}_{mode}" + ("" if tag == "imagenet" else f"_fold{fold}") + ".npy")
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        model = build_model(pretrained=(tag == "imagenet")).to(device)
        if tag != "imagenet":
            ck = torch.load(os.path.join(RUNS, f"{tag}_fold{fold}", "best.pt"), map_location=device)
            if "features" in ck:
                model.features.load_state_dict(ck["features"])
            else:
                model.load_state_dict(ck["model"])
        np.save(path, embed_all(model, load_gray(mode), device))
    F = np.load(path)
    return F / np.linalg.norm(F, axis=1, keepdims=True)


def fit_writer_svm(F, meta, w, neg, n_ref):
    """SVM for writer w: their first n_ref genuine (positives) vs other writers' genuine (negatives).
    Returns the fitted SVM and the meta indices of w's questioned signatures."""
    g = meta[meta.writer == w]
    is_ref = (g.label.values == 0) & (g["index"].values <= n_ref)
    pos = F[g.index.values[is_ref]]
    X = np.vstack([pos, neg])
    y = np.r_[np.ones(len(pos)), np.zeros(len(neg))]
    svm = SVC(kernel="rbf", C=1.0, gamma="scale", class_weight={1: len(neg) / len(pos), 0: 1.0})
    return svm.fit(X, y), g.index.values[~is_ref]


def score_writers(F, meta, writers, neg, n_ref):
    ys, ss, per_writer = [], [], []
    for w in writers:
        svm, quest = fit_writer_svm(F, meta, w, neg, n_ref)
        s = -svm.decision_function(F[quest])            # high = looks NOT like this writer = forged
        yq = meta.label.values[quest]
        ys.append(yq)
        ss.append(s)
        per_writer.append(eer_threshold(yq, s)[1])
    return np.concatenate(ys), np.concatenate(ss), per_writer

## Grad-CAM overlay

In [8]:
def overlay(gray, cam):
    heat = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)[..., ::-1]
    base = np.repeat(gray[..., None], 3, axis=2).astype(np.float32)
    return np.uint8(0.55 * base + 0.45 * heat)